Need to make scaling method work with subset, or pass it the tensor of images instead, or could just scale once since all methods other than normalize are just functions.

subset[0] is one gal -> (img, flag)

In [2]:
import gc
import os
import sys
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
from astropy.io import fits
from sklearn.model_selection import KFold
from torch import nn
from torch.utils.data import Subset
from torchvision import transforms
from torchvision.transforms import Normalize

parent_dir = os.path.abspath(os.path.join(os.path.dirname('utils.py'), '..'))
sys.path.append(parent_dir)
import utils as ut  # noqa: E402

In [3]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT_FEATURES = ['FLAG']
SEED = 42
BEST_ACC = 0

# HYPERPARAMETERS
INPUT_SIZE = 67
EPOCHS = 200
NUM_TRIALS = 20
PATIENCE = 20
OPTIMIZE = False
band_choice = 'norm'  # norm, sub, div
SCALER = 'tanh'
N_SPLITS = 5

In [4]:
if OPTIMIZE:
    COMMENT = input('Enter OPTIMIZATION comment: ')

if band_choice == 'norm':
    BANDS = ['f150w', 'f277w', 'f444w']
elif band_choice == 'div':
    BANDS = ['f150w/f444w', 'f277w/f444w']
elif band_choice == 'sub':
    BANDS = ['f444w-f150w', 'f444w-f277w']

In [5]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

if SEED:
    ut.set_seeds(SEED)

## Prepare Data

In [6]:
cwd = Path.cwd()
cutout_dir = cwd.parent / 'data' / 'cutouts'
model_dir = cwd.parent / 'models'

In [7]:
# LOAD DATA INTO DF

cols = {'idx': [], 'images': [], 'header': []}

for folder in cutout_dir.iterdir():
    if folder.is_dir():
        cols['idx'].append(folder.stem)

        temp_df = ut.fits_df(folder)
        temp_df.sort_values(by='BAND')

        # load images
        temp_imgs = {}
        for _, row in temp_df.iterrows():
            hdul = fits.open(row['FILEPATH'])
            data = hdul[0].data.astype('float32')
            header = hdul[0].header
            temp_imgs[row['BAND']] = data
            hdul.close()
        cols['images'].append(temp_imgs)
        cols['header'].append(header)

# turn dict into df and expand header columns
df = pd.DataFrame(cols)
to_expand = ['header', 'images']
for col in to_expand:
    temp_df = pd.DataFrame(df[col].tolist(), index=df.index)
    df = df.join(temp_df)

df.drop(columns=['header', 'BAND', 'images'], inplace=True)

# shuffle
df = df.sample(frac=1, random_state=SEED) if SEED else df.sample(frac=1)

In [8]:
# GET SUBTRACTED AND DIVIDED BANDS IF NEEDED

if band_choice != 'norm':

    def calculate_ratios(row, sigma=5):
        for numerator in numers:
            num = row[numerator]
            den = row[denominator]
            ratio = ut.masked_ratio(num, den)
            row[numerator + '/' + denominator] = ut.clip_extremes(ratio, sigma)
        return row

    numers = ['f150w', 'f277w']
    denominator = 'f444w'

    for numerator in numers:
        df[denominator + '-' + numerator] = df[denominator] - df[numerator]

    df = df.apply(calculate_ratios, axis=1)

In [9]:
# STACK AND RESIZE BANDS

resize_transform = transforms.Resize(
    INPUT_SIZE, transforms.InterpolationMode.BILINEAR
)

df['images'] = df[BANDS].values.tolist()
df['images'] = [resize_transform(torch.as_tensor(img)) for img in df['images']]

C:\Users\harty\AppData\Local\Temp\ipykernel_23028\3055811126.py:8: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  df['images'] = [resize_transform(torch.as_tensor(img)) for img in df['images']]


In [10]:
# EXTRACT X, y, and i from data

data_df = df[df['TABLE'] != 'test']

X_data = torch.stack(data_df['images'].tolist())
y_data = torch.tensor(data_df['FLAG'].tolist())
i_data = torch.tensor(data_df['idx'].astype(int).tolist())

In [11]:
X_data.shape

torch.Size([438, 3, 67, 67])

In [12]:
# SCALE DATA


def scale_data(data, type='tanh'):

    if type == 'log1p':
        data = torch.log1p(data)
    elif type == 'pytorch_normalize':
        scaler = Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
        data = scaler(data)
    elif type == 'arcsinh':
        data = torch.asinh(data)
    elif type == 'tanh':
        data = torch.tanh(data)
    else:
        raise ValueError(f'{type} is not a valid scaling function')

    return data


X_data_scaled = scale_data(X_data, type=SCALER)
ut.analyze_data(X_data, axis=(0, 2, 3), name='Unscaled data:')
print('\n')
ut.analyze_data(X_data_scaled, axis=(0, 2, 3), name=f'{SCALER} Scaled data:')

Unscaled data:
Mean:  [0.04243751 0.10102423 0.16987188]
Median:  [0.01113511 0.01979694 0.02893846]
St. Dev:  [0.17126837 0.3822131  0.7546609 ]
Max:  [17.483625 34.802776 80.58712 ]
Min:  [-0.06790778 -0.0484455  -0.17565338]
Length:  438


tanh Scaled data:
Mean:  [0.03739235 0.07838354 0.10648803]
Median:  [0.01113465 0.01979436 0.02893038]
St. Dev:  [0.09199782 0.1575378  0.2064333 ]
Max:  [1. 1. 1.]
Min:  [-0.06780359 -0.04840764 -0.17386886]
Length:  438


In [13]:
# CONVERT TO DATSET

# transforms to apply to inputs
train_transform = transforms.Compose(
    [
        # transforms.RandomHorizontalFlip(p=0.5),
        # transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=(0, 360))
        # transforms.RandomPerspective()
    ]
)

dataset = ut.GalaxyDataset(images=X_data_scaled, ids=i_data, labels=y_data)

In [14]:
# KFOLD TESTING

kfold = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = list(kfold.split(dataset))

first_train_idx, first_test_idx = folds[0]

subset = Subset(dataset, first_train_idx)

## Define Model

In [15]:
# DEFINE MODEL AND HYPERPARAMTER FUNCTIONS


def define_model(trial):
    # get model structure from optuna
    n_conv = trial.suggest_int('n_conv', 1, 3)
    n_fc = trial.suggest_int('n_fc', 1, 3)

    conv_channels = [
        trial.suggest_int(f'n_channels_conv{i}', 1, 32) for i in range(3)
    ]
    conv_ksizes = [
        trial.suggest_int(f'kernel_size_conv{i}', 1, 3) for i in range(3)
    ]
    conv_paddings = [
        trial.suggest_int(f'padding_conv{i}', 1, 3) for i in range(3)
    ]
    conv_dropouts = [
        trial.suggest_float(f'dropout_conv{i}', 0.2, 0.5) for i in range(3)
    ]
    fc_units = [
        trial.suggest_int(f'n_units_fc{i}', 4, 256, step=4) for i in range(3)
    ]
    fc_dropouts = [
        trial.suggest_float(f'dropout_fc{i}', 0.2, 0.5) for i in range(3)
    ]

    layers = []

    # build conv layers
    in_channels = len(BANDS)
    current_spatial_size = INPUT_SIZE

    for i in range(n_conv):
        out_channels = conv_channels[i]
        ksize = conv_ksizes[i]
        pad = conv_paddings[i]
        dropout = conv_dropouts[i]

        layers.extend(
            (
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=ksize,
                    padding=pad,
                    bias=False,
                ),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(),
                nn.Dropout(dropout),
            )
        )

        current_spatial_size = current_spatial_size - ksize + (2 * pad) + 1
        in_channels = out_channels

    flattened_features = in_channels * (current_spatial_size**2)
    layers.append(nn.Flatten())

    # build fc layers
    in_features = flattened_features
    for i in range(n_fc):
        out_features = fc_units[i]
        dropout = fc_dropouts[i]

        layers.extend(
            (
                nn.Linear(in_features, out_features, bias=False),
                nn.BatchNorm1d(out_features),
                nn.ReLU(),
                nn.Dropout(dropout),
            )
        )

        in_features = out_features

    layers.append(nn.Linear(in_features, 2 * len(OUT_FEATURES)))

    return nn.Sequential(*layers)


def get_hyperparameters(trial, model):

    # optimizer and learning rate
    lr = trial.suggest_float('lr', 1e-5, 1e-1, log=True)
    optimizer_name = trial.suggest_categorical(
        'optimizer', ['Adam', 'SGD', 'AdamW']
    )

    kwargs = {'lr': lr}

    if optimizer_name == 'AdamW':
        kwargs['weight_decay'] = trial.suggest_float(
            'adamw_weight_decay', 1e-5, 1e-1, log=True
        )
    elif optimizer_name == 'SGD':
        kwargs['momentum'] = trial.suggest_float('sgd_momentum', 0.0, 0.99)

    optimizer = getattr(torch.optim, optimizer_name)(
        model.parameters(), **kwargs
    )

    # weights and loss function
    w1 = trial.suggest_float('w1', 1, 3)
    w2 = trial.suggest_float('w2', 1, 3)
    weights = torch.tensor([w1, w2]).to(DEVICE)
    loss_func = nn.CrossEntropyLoss(weight=weights)

    # batch size and loaders
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])

    return optimizer, loss_func, batch_size

In [ ]:
# DEFINE OBJECTIVE FUNCTION


def objective(trial):
    global BEST_ACC
    model = define_model(trial).to(DEVICE)
    optimizer, loss_func, batch_size = get_hyperparameters(trial, model)

    kfold = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    fold_accuracies = []

    # Split the dataset into 5 folds
    try:
        for fold, (train_idx, val_idx) in enumerate(kfold.split(dataset)):
            # Create subsets for training and validation
            train_subset = Subset(dataset, train_idx)
            val_subset = Subset(dataset, val_idx)

            if SEED:
                train_loader = torch.utils.data.DataLoader(
                    train_subset,
                    batch_size=batch_size,
                    shuffle=False,
                    drop_last=True,
                )
            else:
                train_loader = torch.utils.data.DataLoader(
                    train_subset,
                    batch_size=batch_size,
                    shuffle=True,
                    drop_last=True,
                )
            val_loader = torch.utils.data.DataLoader(
                val_subset, batch_size=batch_size, shuffle=False
            )

            # early stopper
            stopper = ut.EarlyStopper(patience=PATIENCE, min_delta=0.0005)
            for _ in range(EPOCHS):
                # Training Loop ----------------------------------------------------
                model.train()
                ut.do_epoch(
                    model=model,
                    loader=train_loader,
                    loss_function=loss_func,
                    device=DEVICE,
                    optimizer=optimizer,
                    train=True,
                )

                # Validation Loop ---------------------------------------------------
                model.eval()
                with torch.no_grad():
                    running_loss, correct = ut.do_epoch(
                        model=model,
                        loader=val_loader,
                        loss_function=loss_func,
                        device=DEVICE,
                        train=False,
                    )

                    # compute accuracy
                    accuracy = correct / len(val_subset)
                    if accuracy > BEST_ACC:
                        BEST_ACC = float(accuracy)

                    # track loss once per epoch to check early stopper
                    val_loss = running_loss / len(val_subset)
                    stopper(val_loss)
                    if stopper.early_stop:
                        break

            fold_accuracies.append(accuracy)
            trial.report(accuracy, step=fold)

            # prune bad trials
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        return np.mean(fold_accuracies)

    # clean up memory after every trial
    finally:
        if 'model' in locals():
            del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

## Train Model

In [17]:
# TRAINS MODEL

if OPTIMIZE:
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    # use gaussian process
    sampler = (
        optuna.samplers.GPSampler(seed=SEED)
        if SEED
        else optuna.samplers.GPSampler()
    )

    study = optuna.create_study(
        direction='maximize',
        sampler=sampler,
        pruner=optuna.pruners.MedianPruner(),
    )
    study.optimize(objective, n_trials=NUM_TRIALS, show_progress_bar=True)

    print('Best trial:')
    trial = study.best_trial
    print(f'  Accuracy: {trial.value}')
    print('  Params: ')
    for key, value in trial.params.items():
        print(f'    {key}: {value}')

In [ ]:
# LOG OPTIMIZATION

if OPTIMIZE:
    logged_params = trial.params.copy()
    logged_params['comment'] = COMMENT
    logged_params['bands'] = BANDS
    logged_params['out_features'] = OUT_FEATURES
    logged_params['seed'] = SEED
    logged_params['epochs'] = EPOCHS
    logged_params['image_size'] = INPUT_SIZE
    logged_params['num_trials'] = NUM_TRIALS
    logged_params['patience'] = PATIENCE
    logged_params['accuracy'] = trial.value
    logged_params['best accuracy'] = BEST_ACC
    logged_params['scaling'] = SCALER

    logger = ut.RunLogger(filepath=cwd.parent / 'logs' / 'conv.json')
    logger.log_run(logged_params)